# Data Exploration

This notebook details steps taken to investigate and understand the Food Standards Agency (FSA) API, which details Food Hygiene Recognition Scheme (FHRS) ratings for businesses in the UK.  

[API Documentation Link](https://api.ratings.food.gov.uk/Help).  

This is being completed as a first step to examine the response / output of the API, in order to plan and shape the rest of the project.  

### Initial Test Call - Check Status

The first check is to see if a status `200` is returned from a basic 'get' call.

In [11]:
import requests

BASE_URL = "https://api.ratings.food.gov.uk"
HEADERS = {"x-api-version": "2"} # this header is required, otherwise silently fails (and returns nothing)

response = requests.get(f"{BASE_URL}/Authorities", headers=HEADERS) # hit Authorities endpoint
    # https://api.ratings.food.gov.uk/Help/Api/GET-Authorities-pageNumber-pageSize
    
print(response.status_code)

200


An HTTP response status code of `200` ('OK') confirms a successful call.

### Getting London-specific Authority IDs

FHRS ratings are issued by local authorities. As this project focuses on London businesses only, queries will need to be scoped by London authority IDs.

I therefore need to pull out the relevant authority IDs from the `/Authorities` endpoint response above.   

In [12]:
authorities = response.json()["authorities"]
  # authorities is an array of objects, with each element being a new authority

print(len(authorities)) # how many authorities are returned from the initial call

print(authorities[0]) # inspect a single authority object

363
{'LocalAuthorityId': 197, 'LocalAuthorityIdCode': '760', 'Name': 'Aberdeen City', 'FriendlyName': 'aberdeen-city', 'Url': 'http://www.aberdeencity.gov.uk', 'SchemeUrl': '', 'Email': 'commercial@aberdeencity.gov.uk', 'RegionName': 'Scotland', 'FileName': 'https://ratings.food.gov.uk/OpenDataFiles/FHRS760en-GB.xml', 'FileNameWelsh': None, 'EstablishmentCount': 2202, 'CreationDate': '2010-08-17T15:30:24.87', 'LastPublishedDate': '2026-08-21T00:40:33.257', 'SchemeType': 2, 'links': [{'rel': 'self', 'href': 'https://api.ratings.food.gov.uk/authorities/197'}]}


Returns 363 results - this is roughly in line with expectations, as there are approx. 380 total LAs in the UK.  

An example element looks as follows:   

```
{
    'LocalAuthorityId': 197, 
    'LocalAuthorityIdCode': '760', 
    'Name': 'Aberdeen City', 
    'FriendlyName': 'aberdeen-city', 
    'Url': 'http://www.aberdeencity.gov.uk', 
    'SchemeUrl': '', 
    'Email': 'commercial@aberdeencity.gov.uk', 
    'RegionName': 'Scotland', 
    'FileName': 'https://ratings.food.gov.uk/OpenDataFiles/FHRS760en-GB.xml', 
    'FileNameWelsh': None, 
    'EstablishmentCount': 2202, 
    'CreationDate': '2010-08-17T15:30:24.87', 
    'LastPublishedDate': '2026-08-21T00:40:33.257', 
    'SchemeType': 2, 
    'links': [{'rel': 'self', 'href': 'https://api.ratings.food.gov.uk/authorities/197'}]
}
```

The `RegionName` key may be useful in filtering to London only ... assuming there is a `'London'` Region Name.  

In [ ]:
# store a list of authorities, from the existing authorities list, where the authority is in the London region
london_authorities = [a for a in authorities if a["RegionName"] == "London"]

print(len(london_authorities)) # see how many return
print(london_authorities[0]) # see what one of the returned objects looks like

33
{'LocalAuthorityId': 88, 'LocalAuthorityIdCode': '501', 'Name': 'Barking and Dagenham', 'FriendlyName': 'barking-and-dagenham', 'Url': 'http://www.lbbd.gov.uk/Pages/Home.aspx', 'SchemeUrl': '', 'Email': 'foodsafety@lbbd.gov.uk', 'RegionName': 'London', 'FileName': 'https://ratings.food.gov.uk/OpenDataFiles/FHRS501en-GB.xml', 'FileNameWelsh': None, 'EstablishmentCount': 1459, 'CreationDate': '2010-08-17T15:30:24.87', 'LastPublishedDate': '2026-08-13T00:31:25.723', 'SchemeType': 1, 'links': [{'rel': 'self', 'href': 'https://api.ratings.food.gov.uk/authorities/88'}]}


Filtering by a `RegionName` of `London` worked ok. Crucially, the authority count is now 33 - matching the 33 London boroughs (each being a Local Authority).  

Barking and Dagenham is the Local Authority returned, which (encouragingly), is indeed in London.  

`SchemeType` should be `1` for all London LAs, with a 0-5 star scale (note that Aberdeen City above uses scheme 2 instead).  

In [ ]:
# get unique SchemeType values
scheme_types = set(a["SchemeType"] for a in london_authorities) 
print(scheme_types)

{1}


A return of `{1}` confirms that all 33 boroughs are using the same 0-5* scheme.

With the number of London LA results and a consistent Scheme Type confirmed, I can move on from 'Authority-level' checks, and on to establishment records.  